In [1]:
%cd ..
from claimbuster.adv_transformer.core.utils.flags import FLAGS


/home/adamj/factcheck-podcasts/src


In [2]:
import os
# display current working directory
os.getcwd()

'/home/adamj/factcheck-podcasts/src'

In [4]:
FLAGS.cs_model_dir = "/home/adamj/factcheck-podcasts/src/claimbuster/output/bba/"

In [5]:
from claimbuster.adv_transformer.core.api.api_wrapper import ClaimSpotterAPI
claimspotter = ClaimSpotterAPI()

2023-05-12 20:51:44.014848: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
[nltk_data] Downloading package punkt to /home/adamj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/adamj/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to /home/adamj/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package stopwords to /home/adamj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dependencies Loaded.


2023-05-12 20:51:48.077926: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2023-05-12 20:51:48.135271: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-05-12 20:51:48.135356: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4090 computeCapability: 8.9
coreClock: 2.52GHz coreCount: 128 deviceMemorySize: 23.99GiB deviceMemoryBandwidth: 938.86GiB/s
2023-05-12 20:51:48.135395: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2023-05-12 20:51:48.215998: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublas.so.11
2023-05-12 20:51:48.216145: I tensorflow/stream_execut

In [6]:
sentence_list = [
    'Donald Trump is the 45th President of the United States',
    'I really like cheese',
    'McDonalds earns $10 billion dollars each minute'
]

In [7]:
claimspotter.batch_sentence_query(sentence_list)

2023-05-12 20:52:00.507709: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)
2023-05-12 20:52:00.515205: I tensorflow/core/platform/profile_utils/cpu_utils.cc:114] CPU Frequency: 3187195000 Hz


array([[0.75899322, 0.24100678],
       [0.91082659, 0.08917341],
       [0.03764426, 0.96235574]])

In [8]:
import requests
podcasts = requests.get("http://127.0.0.1:8008/api/podcasts/")
podcasts = podcasts.json()

In [9]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy' and podcast['language'] == 'en':
                    segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)


218

In [10]:
agent = "ClaimBuster-BBA-(COREF)"

for seg_uuid in segmentation_uuids:
    segmentation = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
    segmentation = segmentation.json()
    no_classifications = True
    # check if any of the utterances in segmentation["utterance_set"] have a classification made by this agent, if so, skip this segmentation
    for utterance in segmentation["utterance_set"]:
        classifications = utterance["classification_set"]
        for classification in classifications:
            if classification["agent"] == agent:
                no_classifications = False
                break
        if not no_classifications:
            break
    if not no_classifications:
        print("skipping", segmentation.get("uuid"))
        continue
    print("processing", segmentation.get("uuid"))
    sentence_list = [utt["text_coref"] if utt.get("text_coref") else utt["text"] for utt in segmentation["utterance_set"]]
    scores = claimspotter.batch_sentence_query(sentence_list)

    for i, segment in enumerate(segmentation["utterance_set"]):
        if segment.get("text_coref"):
            requests.post(f"http://127.0.0.1:8008/api/classifications/{segment['uuid']}/", json={
                "utterance": segment["uuid"],
                "qualifier": "Checkworthiness",
                "category": "Checkworthy",
                "label": str(scores[i][1]),
                "agent": {"PROLIFIC_PID": agent},
            })

skipping 7909535e-dad4-11ed-ba56-00155d8020a1
skipping 7ad3aa86-dad4-11ed-980f-00155d8020a1
skipping 7cac1e2e-dad4-11ed-9e52-00155d8020a1
skipping 7e2630e6-dad4-11ed-9c3e-00155d8020a1
skipping 9a44cada-dad4-11ed-9644-00155d8020a1
skipping 8303cd26-dca5-11ed-bbbf-00155d0d192b
skipping 1fecf512-dca7-11ed-878d-00155d0d192b
skipping 06f1f674-dca8-11ed-936f-00155d0d192b
skipping 4895fc96-dca9-11ed-a241-00155d0d192b
skipping 26458a62-dcfe-11ed-8243-00155d0d192b
skipping 94374920-e456-11ed-a3db-00155d895813
skipping 977e11ee-e457-11ed-92e9-00155d895813
skipping f0c9fe2e-e458-11ed-ad77-00155d895813
skipping 4cb3f5ae-e45a-11ed-8cce-00155d895813
skipping 6692b540-ead2-11ed-acf9-00155dca624a
skipping d6ea0f22-ead3-11ed-8a39-00155dca624a
skipping b0cb8d14-ead5-11ed-9360-00155dca624a
skipping 548db704-eb1e-11ed-a11a-00155db2e9f9
skipping cea0e656-ef8d-11ed-83d9-00155db2ef8b
processing ca3266de-ef8e-11ed-a1b1-00155db2ef8b
skipping 0ac504bc-f0bc-11ed-a5e2-00155d08852a
skipping 9d2a14c6-dad4-11ed-b7b0

## Get Checkworthiness from Factiverse API

In [ ]:
def get_fv_checkworthiness(text, language="en"):
    query = {'lang': language, 'logging':False, 'text':text}
    response = requests.post('https://api.factiverse.no/v1/claim_detection', json=query)
    response = response.json()
    scores = [resp["score"] for resp in response["detectedClaims"]]
    score = 0 if len(scores) == 0 else sum(scores)/len(scores)
    return score


In [ ]:
for podcast in podcasts:
    language = podcast['language'][0:2] if podcast['language'] != "nb" else "no"
    print(podcast['title'], language)
    if language == "no":
        continue
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy':
                    seg_uuid = segmentation['uuid']
                    segmentation = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
                    segmentation = segmentation.json()

                    for segment in segmentation["utterance_set"]:
                        already_classified = False
                        if segment["classification_set"]:
                            for classification in segment["classification_set"]:
                                if classification["qualifier"] == "Checkworthiness" and classification["agent"] == "Factiverse":
                                    already_classified = True
                                    break
                        if already_classified:
                            continue
                            
                        # split the segment by spaces
                        words = segment["text"].split(" ")
                        if len(words) > 2:
                            fv_score = get_fv_checkworthiness(segment["text"], language)
                        else:
                            fv_score = 0

                        requests.post(f"http://127.0.0.1:8008/api/classifications/{segment['uuid']}/", json={
                            "utterance": segment["uuid"],
                            "qualifier": "Checkworthiness",
                            "category": "Checkworthy",
                            "label": str(fv_score),
                            "agent": "Factiverse"
                        })